# Visuals

Purpose: Run and review one-off simulation visual checks from the existing plotting and animation entrypoints.

Role in project: diagnostic visualization notebook

Inputs:
- `experiments.plot_layout`
- `experiments.plot_attack_once`
- `experiments.render_attack_animation`

Outputs:
- script-owned figures/debug files under `results/figures/`, `results/debug/`, and `results/frames/`
- optional notebook display of the generated artifacts

Expected runtime: short for static plots; medium for animation frames.

Notes:
- This notebook calls existing experiment entrypoints and does not duplicate plotting logic.
- Keep interpretation notes short; detailed script behavior belongs in `docs/SCRIPTS.md`.


## Imports And Repo Setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import subprocess
import sys

from IPython.display import Image, Video, display

# Resolve repo root when this notebook is launched from either repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "convoy_sim").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


## Configuration


In [ ]:
NOTEBOOK_NAME = "visuals"
OUTPUT_DIR = PROJECT_ROOT / "results" / "notebook-results" / NOTEBOOK_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_DIR = PROJECT_ROOT / "results" / "figures"
DEBUG_DIR = PROJECT_ROOT / "results" / "debug"
FRAMES_DIR = PROJECT_ROOT / "results" / "frames"

PYTHON = sys.executable
RUN_ANIMATION = False

RUN_ENV = os.environ.copy()
RUN_ENV["PYTHONPATH"] = str(PROJECT_ROOT)
RUN_ENV.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / "mpl-cache"))


def run_experiment(module: str, *args: str) -> None:
    command = [PYTHON, "-m", module, *args]
    print("$", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, env=RUN_ENV, check=True)


def display_existing_image(path: Path, *, width: int = 720) -> None:
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Missing expected image: {path.relative_to(PROJECT_ROOT)}")


## Plan-View Layouts

Runs the static layout figure generator for quick inspection of convoy formations by ship class and value.


In [ ]:
run_experiment("experiments.plot_layout")


In [ ]:
for figure_name in [
    "rect_class.png",
    "rect_value.png",
    "staggered_class.png",
    "staggered_value.png",
]:
    print(figure_name)
    display_existing_image(FIGURES_DIR / figure_name, width=640)


## Static Attack Overlay

Runs the single-attack static overlay used for quick torpedo-ray and hit/miss annotation checks.


In [ ]:
run_experiment("experiments.plot_attack_once")


In [ ]:
display_existing_image(FIGURES_DIR / "attack_once.png", width=720)
print(f"Debug JSON: {(DEBUG_DIR / 'attack_once.json').relative_to(PROJECT_ROOT)}")


## Temporal Attack Frames

Runs the dynamic attack animation/frame demo. This is disabled by default because it writes many frames.


In [ ]:
if RUN_ANIMATION:
    run_experiment("experiments.render_attack_animation")
else:
    print("Set RUN_ANIMATION = True in the configuration cell to render animation frames.")


In [ ]:
demo_dir = FRAMES_DIR / "demo_attack"
for frame_name in ["frame_0000.png", "frame_0300.png", "frame_0600.png"]:
    display_existing_image(demo_dir / frame_name, width=720)

mp4_path = FRAMES_DIR / "demo_attack.mp4"
if mp4_path.exists():
    display(Video(filename=str(mp4_path), embed=True))
else:
    print(f"No MP4 found at {mp4_path.relative_to(PROJECT_ROOT)}")


## Related Visual Notebooks

- `attack_profile_tests.ipynb`: profile-level geometry audits and representative frame previews.
- `torpedo_firing_doctrine_comparison.ipynb`: doctrine-specific torpedo spread visual semantics.
- `attack_manual_verification.ipynb`: broader manual realism checks.
